# 01 - Exploring the MultiModal PlantMetWiki graph

## Research context and motivation

PlantMetWiki is a cross-species metabolic knowledge graph built from WikiPathways, covering
**424 organisms** (plants, fungi, bacteria, and animals) across **2,478 pathways**.
The graph encodes **84,470 biochemical reactions** (Interaction/Conversion nodes) — but only
**4,572 of them (5.4%) have a known catalysing enzyme** assigned. The remaining
**79,898 reactions (94.6%) are 'orphan reactions'**: the chemistry is documented, but no
gene or protein has been linked to it.

On the protein side the picture is equally sparse: **13,682 proteins** are in the graph,
but only **3,456 (25.3%) are annotated as catalysing any reaction**. Of all proteins,
**38.3% lack an ESM sequence embedding** (protein characterisation is incomplete even where
sequences exist).

This gap is the core biological problem this project addresses:

> **Given a metabolic network with known reactions but sparse enzyme assignments, which
> proteins catalyse which reactions?**

This is the **enzyme discovery** or **orphan-reaction assignment** problem. It is framed as
link prediction in the **Conversion → Protein direction**: for a given reaction (Conversion
node), rank all candidate proteins and identify which one catalyses it. The alternative
framing — given a protein, predict which reactions it catalyses (Protein → Conversion) — is
the protein *annotation* problem, which is a different task and less relevant here because
our metabolic networks are reaction-rich but enzyme-sparse: we know *what happens*, we just
do not know *who does it*.

The model must generalise **across species** (taxa split): the val/test organisms are held out
entirely from training, so the model cannot rely on having seen those proteins before. It must
use protein sequence (ESM embeddings) and reaction chemistry (EC class features, substrate/
product metabolite graph) to transfer enzyme–reaction knowledge from characterised to
under-characterised organisms.

---

This notebook explores the processed PlantMetWiki graph (v3 redesign) after RDF parsing and
conversion to PyTorch Geometric `HeteroData`.

It loads:
- `data/processed/nodes.tsv`
- `data/processed/edges.tsv`
- `data/processed/heterodata.pt`

and produces:
- schema summaries and node/edge count tables
- simple graph statistics and annotation-coverage diagnostics
- reproducible figures in `reports/figures/`


In [ ]:
from pathlib import Path
import sys, platform, random

import pandas as pd
import torch
import matplotlib.pyplot as plt

try:
    import torch_geometric
except ImportError:
    torch_geometric = None

plt.rcParams["figure.figsize"] = (8, 5)
plt.rcParams["figure.dpi"] = 120

ROOT = Path.cwd()
if not (ROOT / "data").exists() and (ROOT.parent / "data").exists():
    ROOT = ROOT.parent

DATA_DIR  = ROOT / "data" / "processed"
FIG_DIR   = ROOT / "reports" / "figures"
TABLE_DIR = ROOT / "reports" / "tables"
FIG_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

NODES_PATH  = DATA_DIR / "nodes.tsv"
EDGES_PATH  = DATA_DIR / "edges.tsv"
HETERO_PATH = DATA_DIR / "heterodata.pt"

print(f"Python {sys.version.split()[0]} | Torch {torch.__version__} | PyG {getattr(torch_geometric, '__version__', 'n/a')}")
print(f"ROOT: {ROOT}")
for p in [NODES_PATH, EDGES_PATH, HETERO_PATH]:
    status = "OK" if p.exists() else "MISSING – run pipeline scripts"
    print(f"  {p.name}: {status}")

nodes = pd.read_csv(NODES_PATH, sep="\t")
edges = pd.read_csv(EDGES_PATH, sep="\t")

if HETERO_PATH.exists():
    data = torch.load(HETERO_PATH, weights_only=False)
    print("\n", data)
else:
    data = None
    print("\nheterodata.pt not found. Run:  python scripts/build_pyg.py")

## 1. Inspect the Table Schema

In [ ]:
print("Node columns:", nodes.columns.tolist())
print("Edge columns:", edges.columns.tolist())
print()
display(nodes.head(3))
display(edges.head(3))

## 2. Node type distribution

In [ ]:
node_counts = (
    nodes["node_type"].value_counts()
    .rename_axis("node_type").reset_index(name="count")
)
print(node_counts.to_string(index=False))
node_counts.to_csv(TABLE_DIR / "node_type_counts.tsv", sep="\t", index=False)

fig, ax = plt.subplots()
ax.barh(node_counts["node_type"], node_counts["count"])
ax.set_xlabel("Count")
ax.set_title("Node type distribution")
plt.tight_layout()
fig.savefig(FIG_DIR / "node_type_distribution.png")
plt.show()

## 3. Edge type distribution

In [ ]:
edge_counts = (
    edges.groupby(["src_type", "rel", "dst_type"])
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
)
print(edge_counts.to_string(index=False))
edge_counts.to_csv(TABLE_DIR / "edge_type_counts.tsv", sep="\t", index=False)

top = edge_counts.head(15).copy()
top["label"] = top["src_type"] + " -[" + top["rel"] + "]-> " + top["dst_type"]

fig, ax = plt.subplots(figsize=(9, 6))
ax.barh(top["label"][::-1], top["count"][::-1])
ax.set_xlabel("Count")
ax.set_title("Top 15 edge types")
plt.tight_layout()
fig.savefig(FIG_DIR / "edge_type_distribution.png")
plt.show()

### 3.1 What each edge type means — and the `is_about` edge explained

The 24 edge types group into four semantic categories:

---

**Group 1 — Task edges**

| Relation | Meaning |
|----------|--------|
| `catalyzes` | ⭐ Target: enzyme (Protein) catalyses a reaction (Interaction/Conversion). The link-prediction objective. |
| `catalyzed_by` | Reverse of `catalyzes`. Only training-positive edges enter the MP graph; val/test never included. **Removed in v6+ baseline** — it kept the 2-hop P→C→P training shortcut that let the model memorise training pairs. |

**Group 2 — Reaction structure / chemistry**

| Relation | Meaning |
|----------|--------|
| `participants` | Metabolite (or sub-Interaction) takes part in a reaction. Main metabolite→reaction connectivity. |
| `source` / `target` | Directed role: reactant vs. product. |
| `is_part_of` (→ Interaction) | Metabolite or sub-reaction is a component of a reaction. Legitimate chemistry — do NOT remove. |
| `is_part_of` (→ Pathway) | Entity belongs to a metabolic pathway. **Removed by default** (`remove_is_part_of=True`): pathway co-membership inflates AUC to ≥0.98 without biological learning. |

**Group 3 — Gene / organism structure**

| Relation | Meaning |
|----------|--------|
| `encodes` / `encoded_by` | Gene (GeneProduct) → Protein. Derived from `TranscriptionTranslation` nodes in the v3 redesign. |
| `organism` | Maps a protein/gene/metabolite to its plant species (NCBITaxon). Used to define the taxa split. |

**Group 4 — `is_about`: WikiPathways entity identity links**

> ⚠️ **This is the most important edge type to understand correctly.**

`is_about` comes from the WikiPathways RDF predicate `wp:isAbout` (line 71 of
`scripts/rdf_to_typed_tables.py`). It is **not** a similarity, homology, or shortcut
relationship. It is a **provenance / identity link** built into WikiPathways' own GPML data model.

**Background:** In WikiPathways, every entity in a pathway diagram has two URIs:
1. A **canonical external ID** — e.g. `https://identifiers.org/uniprot/Q7G9P4`
2. A **pathway-local DataNode** — e.g. `.../Pathway/PC1/DataNode/AT2G27150_MONOMER`

`wp:isAbout` states: *"this canonical entity (Q7G9P4) is the real-world object depicted by
this DataNode (AT2G27150_MONOMER) in pathway PC1."*

After the v3 graph redesign reclassifies DataNodes into typed nodes, both the canonical ID and
the DataNode alias have the same node type — producing `is_about` edges between same-typed nodes.

**What each `is_about` type actually connects (verified by graph inspection):**

| Edge type | Count | What it connects | A shortcut? |
|-----------|------:|-----------------|-------------|
| `(Interaction, is_about, Interaction)` | 29,740 | ALL connect the **same canonical reaction** under different pathway URIs for different organisms: `RXN_18780_in_PC1 → RXN_18780_in_PC3`. Same reaction ID, different node indices because it appears in multiple organisms' pathway files. | **No** — this is the **only** structural bridge between cross-organism instances of the same reaction. |
| `(Protein, is_about, Protein)` | 9,932 | UniProt canonical protein → PlantCyc DataNode alias (e.g. `Q7G9P4 → AT2G27150_MONOMER`). Same protein, two database IDs. | **No** — DataNode destinations have **zero** `catalyzes` edges; no label leakage. |
| `(Metabolite, is_about, Metabolite)` | 12,812 | Same compound (InChIKey) → DataNode alias. | Identity, not similarity. |
| `(GeneProduct, is_about, GeneProduct)` | 4,580 | Same gene product, different pathway URIs. | Identity. |
| `(Pathway, is_about, Pathway)` | 2,478 | Pathway diagram instance → canonical pathway record. | Identity. |

**Why `is_about` edges are load-bearing for cross-species generalisation:**

The `(Interaction, is_about, Interaction)` edges are the **only** structural connection between
different organisms' versions of the same biochemical reaction. Under the taxa (organism-holdout)
split:
- A training protein from *A. thaliana* catalyses `RXN_18780` → represented as node A in the graph
- A held-out test protein from *G. max* catalyses `RXN_18780` → represented as node B
- Nodes A and B are different PyG node indices. Without `is_about`, they are **completely
  disconnected** — the GNN cannot know they are the same reaction.
- With `is_about` (A ↔ B), the GNN learns a shared reaction representation across organisms.

**Ablation evidence:** Removing all `is_about` edges (v7 baseline, 2026-07-03) caused
catastrophic failure. AUC inverted to 0.188 by epoch 10; test P-H@50 = 0.003 (below random
baseline of 0.59%). GNN training destroyed pre-trained ESM-C embedding quality rather than
improving it. See `reports/BASELINE_STRATEGY.md §1d` for the full failure analysis.

**Conclusion:** Treat `is_about` edges like `participants` and `source/target` —
load-bearing structural edges from the WikiPathways data model. Do not remove them.


## 4. Degree statistics per node type

In [ ]:
all_endpoints = pd.concat([
    edges[["src", "src_type"]].rename(columns={"src": "node_id", "src_type": "node_type"}),
    edges[["dst", "dst_type"]].rename(columns={"dst": "node_id", "dst_type": "node_type"}),
])
degrees = all_endpoints.groupby(["node_type", "node_id"]).size().reset_index(name="degree")

degree_stats = (
    degrees.groupby("node_type")["degree"]
    .describe(percentiles=[0.5, 0.75, 0.9, 0.95])
    .round(1)
)
display(degree_stats)
degree_stats.to_csv(TABLE_DIR / "degree_stats.tsv", sep="\t")

## 5. Graph composition, structure & organism (taxonomy) coverage

In [ ]:
from IPython.display import Markdown

# ── Graph composition ──────────────────────────────────────────────────────────
def safe_count(node_type):
    """0 if a node type no longer exists (e.g. DataNode/External, removed by the
    v3 graph redesign in Section 9) rather than crashing on re-run."""
    m = node_counts.loc[node_counts["node_type"] == node_type, "count"]
    return int(m.values[0]) if len(m) else 0


total_nodes  = node_counts["count"].sum()
n_interaction = safe_count("Interaction")
n_metabolite  = safe_count("Metabolite")
n_protein     = safe_count("Protein")
n_geneproduct = safe_count("GeneProduct")
n_datanode    = safe_count("DataNode")
n_pathway     = safe_count("Pathway")
n_organism    = safe_count("Organism")
n_external_nodes = safe_count("External")
pct_interaction  = 100 * n_interaction / total_nodes

# ── Edge structure ─────────────────────────────────────────────────────────────
total_edges        = int(edge_counts["count"].sum())
n_edge_types       = len(edge_counts)
directional_edges  = int(edge_counts[edge_counts["rel"].isin(["source", "target"])]["count"].sum())
participants_edges = int(edge_counts[edge_counts["rel"] == "participants"]["count"].sum())

# ── Degree statistics ──────────────────────────────────────────────────────────
met_mean = degree_stats.loc["Metabolite",  "mean"];  met_std = degree_stats.loc["Metabolite",  "std"];  met_max  = int(degree_stats.loc["Metabolite",  "max"])
int_mean = degree_stats.loc["Interaction", "mean"];  int_std = degree_stats.loc["Interaction", "std"]
pw_mean  = degree_stats.loc["Pathway",     "mean"];  pw_max  = int(degree_stats.loc["Pathway",     "max"])

# ── Organism (taxonomy) coverage ─────────────────────────────────────────────────
# Species assignment comes from a separate companion RDF file
# (all_gpml_taxonomy_extra-*.ttl.gz) which links Protein/GeneProduct/Metabolite/DataNode
# nodes directly to NCBITaxon organisms (see CLAUDE.md "Taxonomy (species) assignment").
# The core RDF dump's own wp:organism triples are a generic Pathway-level placeholder
# (NCBITaxon_131567, "cellular organisms") and carry no species signal, so they are
# NOT used and the (Pathway, organism, Organism) edge type no longer exists.
# DataNode is filtered out of this list if it no longer exists as a node type (Section 9).
ORGANISM_TARGET_TYPES = [t for t in ["Protein", "GeneProduct", "Metabolite", "DataNode"]
                          if t in set(node_counts["node_type"])]
FALLBACK_TAXON_URI = "http://purl.obolibrary.org/obo/NCBITaxon_33090"  # Viridiplantae, species unspecified

org_edges = edges[edges["rel"] == "organism"]

taxa_per_node = org_edges.groupby("src")["dst"].agg(set)
fallback_nodes = set(taxa_per_node[taxa_per_node.apply(lambda s: s == {FALLBACK_TAXON_URI})].index)
multi_label_nodes = set(taxa_per_node[taxa_per_node.apply(len) > 1].index)

n_target_total = sum(safe_count(t) for t in ORGANISM_TARGET_TYPES)
n_fallback   = len(fallback_nodes)
n_multi      = len(multi_label_nodes)
n_covered    = taxa_per_node.index.nunique() - n_fallback
pct_covered  = 100 * n_covered / n_target_total
pct_fallback = 100 * n_fallback / n_target_total
pct_multi    = 100 * n_multi / n_target_total
n_organisms_real = org_edges.loc[org_edges["dst"] != FALLBACK_TAXON_URI, "dst"].nunique()

# Coverage broken down by node type
org_coverage = pd.DataFrame([
    {
        "node_type": t,
        "total":   safe_count(t),
        "covered": len(set(org_edges.loc[org_edges["src_type"] == t, "src"]) - fallback_nodes),
    }
    for t in ORGANISM_TARGET_TYPES
])
org_coverage["fallback"]    = org_coverage["total"] - org_coverage["covered"]
org_coverage["pct_covered"] = (100 * org_coverage["covered"] / org_coverage["total"]).round(1)
display(org_coverage)
org_coverage.to_csv(TABLE_DIR / "organism_coverage_by_type.tsv", sep="\t", index=False)

# Most common real (non-fallback) species by number of assigned nodes
top_taxa = (
    org_edges[org_edges["dst"] != FALLBACK_TAXON_URI]
    .assign(taxon=lambda d: d["dst"].str.rsplit("_", n=1).str[-1])
    .groupby("taxon").size().reset_index(name="n_nodes")
    .sort_values("n_nodes", ascending=False)
    .head(10)
)
display(top_taxa)

# ── External node discrepancy ──────────────────────────────────────────────────
n_ext_in_edgelist = int(degree_stats.loc["External", "count"]) if "External" in degree_stats.index else 0
n_ext_dropped     = n_ext_in_edgelist - n_external_nodes

# ── Render ─────────────────────────────────────────────────────────────────────
display(Markdown(f"""
### Graph Composition

The graph contains **{total_nodes:,} nodes** across {node_counts["node_type"].nunique()} node types and **{total_edges:,} edge instances**
across {n_edge_types} relation types.

**`Interaction` nodes dominate** ({n_interaction:,} nodes, {pct_interaction:.0f}% of the graph).
In WikiPathways, an `Interaction` represents a biochemical event (reaction, catalysis, binding) —
it is the central entity for link prediction.
The biologically meaningful entities are comparatively small:
{n_metabolite:,} Metabolites, {n_protein:,} Proteins, {n_geneproduct:,} GeneProducts,
{n_datanode:,} DataNodes, {n_pathway:,} Pathways, and {n_organism:,} Organisms (NCBITaxon species).
(If this is a re-run after Section 9's graph redesign, `DataNode`/`External` will read 0 -- they
no longer exist as node types.)

---

### Edge Structure

Most edge types come in **symmetric pairs** (e.g. `Interaction –[participants]→ Metabolite` and
`Metabolite –[is_part_of]→ Interaction` carry identical counts), encoding the same relationship
bidirectionally from the RDF source.

The **directional edges** (`source` / `target`, {directional_edges:,} total) are the most informative
for the link-prediction task: they encode which entities feed into a reaction and what a reaction
produces. Together with `participants` ({participants_edges:,} edges), these form the natural
supervision signal for predicting missing enzyme–reaction and metabolite–reaction links.

---

### Degree Distribution

**Metabolites are extreme hubs** (mean degree {met_mean:.1f}, std {met_std:.1f}, max {met_max:,}).
This heavy-tailed distribution reflects a small number of universal cofactors (ATP, NADH, water, CoA)
that participate in thousands of reactions. This skew requires careful handling during train/val/test
splitting to avoid data leakage through shared hub nodes.

**Interactions are relatively uniform** (mean {int_mean:.1f}, std {int_std:.1f}): most reactions
involve a small, fixed number of participants, which is biochemically expected.

**Pathways** show moderate variability (mean {pw_mean:.1f}, max {pw_max}): a few large hub pathways
(e.g. central carbon metabolism) coexist with many small specialised ones.

---

### Organism (Taxonomy) Coverage

Species information now lives in a **separate companion file**
(`all_gpml_taxonomy_extra-*.ttl.gz`), assigned directly to **Protein, GeneProduct, Metabolite,
and DataNode** nodes via `(node, organism, Organism)` edges built by
`scripts/rdf_to_typed_tables.py`. The core RDF dump's pathway-level `wp:organism` triples are a
generic `NCBITaxon_131567` ("cellular organisms") placeholder with no species signal and are not
included in the graph at all.

Out of **{n_target_total:,}** {' / '.join(ORGANISM_TARGET_TYPES)} nodes:

- **{n_covered:,} ({pct_covered:.0f}%)** have a real species assignment from the taxonomy-extra
  file, spanning **{n_organisms_real}** distinct taxa (table of top 10 above).
- **{n_fallback:,} ({pct_fallback:.0f}%)** have no entry in the taxonomy-extra file and instead
  receive a synthetic **"unknown organism"** fallback label, `NCBITaxon_33090`
  (*Viridiplantae*, i.e. green plants with species unspecified). This fallback is a deliberate
  design choice (`scripts/rdf_to_typed_tables.py`, `FALLBACK_TAXON`) so that every
  taxonomy-target node has at least one `organism` edge.
- **{n_multi:,} ({pct_multi:.1f}%)** nodes are assigned **more than one** taxon (multi-label
  organism assignment) — allowed in the graph (e.g. an enzyme conserved across several annotated
  species).

Coverage varies by node type (table above): Proteins and GeneProducts are covered far more often
than Metabolites and DataNodes, reflecting which entities WikiPathways annotates with species
directly.

**Implication for splitting**: `02_build_splits.ipynb` builds an organism-held-out ("taxa") split
using these `organism` edges. Both the fallback-labelled nodes (`NCBITaxon_33090`, "unknown
organism") and the multi-label nodes are **excluded from held-out (val/test) organism
assignment** — a node whose only species signal is "unspecified green plant", or that is shared
across several species, cannot be cleanly attributed to a single held-out organism, so its edges
always stay in train.

---

### Data Integrity Note

The degree statistics computed from `edges.tsv` reveal **{n_ext_in_edgelist:,} distinct External URIs**
referenced in the edge list, yet `heterodata.pt` contains only **{n_external_nodes} External nodes**.
The {n_ext_dropped:,} discarded entries are `is_about` links pointing to external database identifiers
(ChEBI, UniProt, etc.) that were not present in `nodes.tsv` and were therefore silently dropped by
`build_pyg.py`. If cross-database identifiers are needed as node features in future, this step of the
pipeline will need revisiting. (Section 9 acts on this directly: `External` is dropped from the
graph entirely.)
"""))

## 6. Interaction subtypes in the source RDF

`rdf_to_typed_tables.py` collapses several WikiPathways ontology classes into a single
`Interaction` node type (see `CLAUDE.md`, "Key design decisions"). The full priority-ordered
mapping (`CLASS_URI_TO_TYPE`) sends `wp:Interaction`, `gpml:Interaction`,
`wp:DirectedInteraction`, `wp:Conversion`, `wp:Catalysis`, `wp:Inhibition`, `wp:Stimulation`,
`wp:Binding`, `wp:ComplexBinding`, and `wp:TranscriptionTranslation` all to `Interaction`.

These classes have very different biological meanings (from the WikiPathways RDF ontology):

| WP class | Meaning | `source` / `target` / `participants` |
|---|---|---|
| `wp:Conversion` | A biochemical **reaction**: substrate(s) -> product(s) | Metabolites |
| `wp:Catalysis` | An **enzyme catalyses a reaction**: `source` = catalysing Protein, `target`/`participants` = the `Conversion` it controls | Protein + Interaction (the Conversion) |
| `wp:TranscriptionTranslation` | Gene expression step: GeneProduct -> Protein | GeneProduct, Protein |
| `wp:Inhibition` / `wp:Stimulation` | Regulatory edge: `source` = regulator (usually a Metabolite), `target` = the regulated `Conversion` | Metabolite + Interaction |
| `wp:Binding` / `wp:ComplexBinding` | Complex formation | Protein / DataNode |
| `wp:Interaction` / `gpml:Interaction` / `wp:DirectedInteraction` | Generic base classes carried by almost every node above | - |

To find out which of these is "the reaction" for missing-reaction prediction (CLAUDE.md:
*predict missing enzyme-reaction or metabolite-reaction links*), we recover the original
`rdf:type` assertions from `data/interim/reactions.ttl` for every node currently typed
`Interaction`, restricted to these "leaf" biological classes (excluding the generic base
classes, which almost every node also carries).

In [ ]:
INTERACTION_SUBTYPE_CACHE = TABLE_DIR / "interaction_subtypes.tsv"

WP = "http://vocabularies.wikipathways.org/wp#"
RDF_TYPE = "http://www.w3.org/1999/02/22-rdf-syntax-ns#type"

# "Leaf" subtypes that distinguish the biological meaning of an Interaction node.
# wp:Interaction / gpml:Interaction / wp:DirectedInteraction are generic base classes
# carried by almost every Interaction node, so they're excluded here.
INTERACTION_SUBTYPE_CLASSES = {
    WP + "Conversion": "Conversion",
    WP + "Catalysis": "Catalysis",
    WP + "TranscriptionTranslation": "TranscriptionTranslation",
    WP + "Inhibition": "Inhibition",
    WP + "Stimulation": "Stimulation",
    WP + "Binding": "Binding",
    WP + "ComplexBinding": "ComplexBinding",
}

interaction_ids = set(nodes.loc[nodes["node_type"] == "Interaction", "node_id"])

if INTERACTION_SUBTYPE_CACHE.exists():
    subtypes = pd.read_csv(INTERACTION_SUBTYPE_CACHE, sep="\t")
else:
    from rdflib import Graph, URIRef
    rttl = ROOT / "data" / "interim" / "reactions.ttl"
    g = Graph()
    g.parse(rttl.as_posix(), format="turtle")
    print(f"Loaded {len(g):,} triples from {rttl}")

    rows = [
        {"node_id": str(s), "subtype": INTERACTION_SUBTYPE_CLASSES[str(o)]}
        for s, _, o in g.triples((None, URIRef(RDF_TYPE), None))
        if str(o) in INTERACTION_SUBTYPE_CLASSES and str(s) in interaction_ids
    ]
    subtypes = pd.DataFrame(rows)
    subtypes.to_csv(INTERACTION_SUBTYPE_CACHE, sep="\t", index=False)

n_interaction_total = len(interaction_ids)
n_labeled = subtypes["node_id"].nunique()
n_generic = n_interaction_total - n_labeled
print(f"Interaction nodes: {n_interaction_total:,}")
print(f"  with a recovered 'leaf' subtype:    {n_labeled:,} ({100 * n_labeled / n_interaction_total:.1f}%) "
      f"-- semantic wp:Interaction nodes")
print(f"  without one (generic/diagram-only): {n_generic:,} ({100 * n_generic / n_interaction_total:.1f}%) "
      f"-- mostly gpml:Interaction diagram connectors")

In [ ]:
subtype_counts = (
    subtypes["subtype"].value_counts()
    .rename_axis("subtype").reset_index(name="count")
)
subtype_counts["pct_of_labeled"] = (100 * subtype_counts["count"] / n_labeled).round(1)
print(subtype_counts.to_string(index=False))
subtype_counts.to_csv(TABLE_DIR / "interaction_subtype_counts.tsv", sep="\t", index=False)

fig, ax = plt.subplots()
ax.barh(subtype_counts["subtype"], subtype_counts["count"])
ax.set_xlabel("# Interaction nodes")
ax.set_title("Interaction subtypes recovered from rdf:type")
ax.invert_yaxis()
plt.tight_layout()
plt.savefig(FIG_DIR / "interaction_subtypes.png")
plt.show()

In [ ]:
# What do the source/target/participants edges of each subtype actually connect to?
subtype_lookup = subtypes.drop_duplicates("node_id").set_index("node_id")["subtype"]

dir_edges = edges[
    (edges["src_type"] == "Interaction") & edges["rel"].isin(["source", "target", "participants"])
].copy()
dir_edges["subtype"] = dir_edges["src"].map(subtype_lookup).fillna("(generic, no leaf subtype)")

neighbor_breakdown = (
    dir_edges.groupby(["subtype", "rel", "dst_type"]).size()
    .reset_index(name="count")
    .sort_values(["subtype", "count"], ascending=[True, False])
)
neighbor_breakdown.to_csv(TABLE_DIR / "interaction_subtype_neighbors.tsv", sep="\t", index=False)
display(neighbor_breakdown)

In [ ]:
from IPython.display import Markdown

n_conversion = int(subtype_counts.loc[subtype_counts["subtype"] == "Conversion", "count"].iloc[0])
n_catalysis  = int(subtype_counts.loc[subtype_counts["subtype"] == "Catalysis", "count"].iloc[0])
n_transcr    = int(subtype_counts.loc[subtype_counts["subtype"] == "TranscriptionTranslation", "count"].iloc[0])
n_inhib      = int(subtype_counts.loc[subtype_counts["subtype"] == "Inhibition", "count"].iloc[0])
n_stim       = int(subtype_counts.loc[subtype_counts["subtype"] == "Stimulation", "count"].iloc[0])
n_bind       = int(subtype_counts.loc[subtype_counts["subtype"].isin(["Binding", "ComplexBinding"]), "count"].sum())

display(Markdown(f"""
### Reflection: which "Interaction" subtype is *the* reaction?

Of the {n_interaction_total:,} `Interaction` nodes, only **{n_labeled:,} ({100*n_labeled/n_interaction_total:.0f}%)**
carry a "leaf" biological subtype from `rdf:type`; the other **{n_generic:,} ({100*n_generic/n_interaction_total:.0f}%)**
are `gpml:Interaction` diagram connectors (x/y/colour/etc.) with almost no `source`/`target`/
`participants` edges -- visual layout artifacts, not biology, contributing essentially nothing to
the learning task.

Among the {n_labeled:,} biologically-typed Interaction nodes:

| Subtype | Count | % | What its `source`/`target`/`participants` connect to |
|---|---:|---:|---|
| **`Conversion`** | {n_conversion:,} | {100*n_conversion/n_labeled:.0f}% | **Metabolites only** -- substrates (`source`), products (`target`), all reactants/products (`participants`) |
| **`Catalysis`** | {n_catalysis:,} | {100*n_catalysis/n_labeled:.0f}% | `source` = catalysing **Protein**, `target`/`participants` = the **`Conversion`** it controls |
| `TranscriptionTranslation` | {n_transcr:,} | {100*n_transcr/n_labeled:.0f}% | GeneProduct -> Protein (gene expression, not metabolism) |
| `Inhibition` | {n_inhib:,} | {100*n_inhib/n_labeled:.0f}% | `source` = regulating Metabolite, `target` = the regulated `Conversion` |
| `Stimulation` | {n_stim:,} | {100*n_stim/n_labeled:.0f}% | same pattern as `Inhibition`, much rarer |
| `Binding`/`ComplexBinding` | {n_bind:,} | {100*n_bind/n_labeled:.0f}% | complex formation (Protein/DataNode), negligible |

**`Conversion` is the reaction.** It is the only subtype whose `source`/`target`/`participants`
edges connect exclusively to `Metabolite` nodes -- exactly the **metabolite-reaction links**
CLAUDE.md describes as the central prediction target. A "missing reaction" is most naturally
framed as predicting a missing `(Metabolite, participants/source/target, Conversion)` edge, or
predicting a whole new `Conversion` node and its metabolite edges.

**`Catalysis` is the enzyme-reaction link, not a reaction itself.** Every `Catalysis` node forms a
fixed pattern: `Protein --source--> Catalysis --target--> Conversion` (plus reciprocal
`participants` edges to both). This *is* the **enzyme-reaction link** from CLAUDE.md -- for an
enzyme-reaction prediction task, the natural framing is "does `Protein` X catalyse `Conversion`
Y", which can be read off (or collapsed into a direct `Protein -catalyzes-> Conversion` edge)
from these `Catalysis` nodes.

**Everything else is a different biological question.** `TranscriptionTranslation`
({n_transcr:,} nodes) is about gene expression, not metabolism. `Inhibition`/`Stimulation`
({n_inhib + n_stim:,} nodes combined) are regulatory edges where a metabolite modulates a
*Conversion* rather than participating in it -- biologically interesting, but a different
prediction target (regulation, not stoichiometry). `Binding`/`ComplexBinding` is negligible
({n_bind} nodes).

**Implication for the pipeline**: currently `Conversion`, `Catalysis`, `Inhibition`,
`Stimulation`, `TranscriptionTranslation`, `Binding`/`ComplexBinding`, and ~{n_generic:,}
diagram-only nodes are *all* the same `Interaction` node type, and their
`source`/`target`/`participants` edges are indistinguishable in `edges.tsv` / `heterodata.pt`. A
`(Metabolite, participants, Interaction)` edge could mean "is a substrate/product of this
reaction" *or* "inhibits this reaction" -- two very different biological claims a link-prediction
model currently cannot tell apart. The recovered `subtype` label (written to
`reports/tables/interaction_subtypes.tsv`) is a natural candidate for an extra `nodes.tsv`
column, so `04_link_prediction.ipynb` can:
- focus the missing-reaction target on `(Metabolite, *, Conversion)` and
  `(Protein, *, Catalysis)` / `(Catalysis, *, Conversion)` edges,
- optionally drop the {n_generic:,} diagram-only `Interaction` nodes entirely, and
- keep `Inhibition` / `Stimulation` / `TranscriptionTranslation` as separate (or excluded) edge
  types rather than mixing them into the reaction-participation signal.
"""))

## 7. Graph schema: nodes, edges, and what each means for modelling

The figure below summarises the heterogeneous graph using the refined `Interaction` subtypes
from Section 6. Each box is a node type with its node count; each arrow is a relation with its
edge count. Colours mark each type's role in the modelling plan (justified in Section 8):

- **green** = entity types to featurise (`03_*_encodings.ipynb`: gene, protein, metabolite)
- **red** = the prediction target (`Catalysis` -- enzyme/reaction assignment)
- **blue** = `Conversion`, the reaction itself: its identity is defined by its `Metabolite`
  participants (context, not the prediction target)
- **grey** = auxiliary interaction subtypes -- real biology, but a different relation than
  "catalyses"; kept as context, not as the split target
- **yellow** = `Pathway` / `Organism` / `External` metadata
- **near-white** = excluded from the learning graph

`Pathway`, `Organism`, and `External` are shown as boxes (with their node counts), but their
connecting edges are *not drawn* -- with ~96k `is_part_of` and ~54k `organism` edges fanning out
from almost every other node type, drawing all of them made the diagram unreadable (overlapping
arrows and counts). The table below the figure gives the per-node-type edge counts instead.
**These edges are not missing from the data**: they are fully present in `heterodata.pt` and
directly drive `02_build_splits.ipynb` -- the pathway-balanced split (Section 6 there) walks
`is_part_of` edges to hold out whole pathways, and the taxa split (Section 5 there) walks
`organism` edges to hold out whole species.

`is_about` edges to `External` are similarly omitted from both the figure and the table -- they
carry no reaction or taxonomy information and have no modelling role yet.

A simplified diagram showing only the nodes and edges directly involved in the `Catalysis`
link-prediction task follows after the table.

In [ ]:
import numpy as np
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch


def ncount(node_type):
    # 0 if the node type no longer exists (e.g. DataNode/External after Section 9's redesign)
    m = node_counts.loc[node_counts["node_type"] == node_type, "count"]
    return int(m.iloc[0]) if len(m) else 0


def nb_count(subtype, rel, dst_type):
    m = neighbor_breakdown[
        (neighbor_breakdown["subtype"] == subtype)
        & (neighbor_breakdown["rel"] == rel)
        & (neighbor_breakdown["dst_type"] == dst_type)
    ]
    return int(m["count"].iloc[0]) if len(m) else 0


def curve_point(p0, p1, rad, t):
    """Point at parameter t along the arc3(rad) connection curve from p0 to p1.

    Used to place edge labels ON the drawn curve rather than on the
    straight-line midpoint, and to stagger labels of edges that share an
    endpoint (via rad/t) so they do not overlap each other or the nodes.
    """
    p0 = np.asarray(p0, dtype=float)
    p1 = np.asarray(p1, dtype=float)
    mid = (p0 + p1) / 2
    d = p1 - p0
    perp = np.array([-d[1], d[0]])
    ctrl = mid + rad * perp
    return (1 - t) ** 2 * p0 + 2 * (1 - t) * t * ctrl + t ** 2 * p1


Metabolite_count  = ncount("Metabolite")
Protein_count     = ncount("Protein")
GeneProduct_count = ncount("GeneProduct")
DataNode_count    = ncount("DataNode")
Pathway_count     = ncount("Pathway")
Organism_count    = ncount("Organism")
External_count    = ncount("External")

# label -> (count, category, (x, y))
node_info = {
    "Metabolite":               (Metabolite_count,  "feature", (0.5, 5.5)),
    "GeneProduct":              (GeneProduct_count, "feature", (0.5, 1.0)),
    "Protein":                  (Protein_count,     "feature", (9.5, 3.0)),
    "DataNode":                 (DataNode_count,    "exclude", (9.5, -1.0)),
    "Conversion":               (n_conversion,      "reaction", (3.7, 5.5)),
    "Catalysis":                (n_catalysis,       "target",   (6.8, 4.5)),
    "TranscriptionTranslation": (n_transcr,         "aux",      (3.7, 1.0)),
    "Inhibition":               (n_inhib,           "aux",      (6.8, 7.0)),
    "Stimulation":              (n_stim,            "aux",      (6.8, 8.6)),
    "Binding /\nComplexBinding": (n_bind,          "aux",      (9.5, 1.0)),
    "Pathway":                  (Pathway_count,     "meta",     (3.7, 9.7)),
    "Organism":                 (Organism_count,    "meta",     (12.2, 1.0)),
    "External":                 (External_count,    "meta",     (12.2, 4.5)),
    "Other Interaction\n(diagram-only)": (n_generic, "exclude", (0.5, -1.5)),
}

# (src, dst, edge label, count, role, rad, t)
# rad = arc curvature of the connector; t = label position along that curve
# (0 = src, 1 = dst). rad/t are tuned per edge so curves and labels that share
# an endpoint fan out instead of stacking on top of each other.
edge_list = [
    ("Conversion", "Metabolite", "participants\n(n={:,})".format(
     nb_count("Conversion", "participants", "Metabolite")), "reaction", 0.18, 0.5),
    ("Catalysis", "Protein", "source\n(n={:,})".format(
     nb_count("Catalysis", "source", "Protein")), "target", 0.18, 0.5),
    ("Catalysis", "Conversion", "target\n(n={:,})".format(
     nb_count("Catalysis", "target", "Interaction")), "target", 0.30, 0.5),
    ("Inhibition", "Metabolite", "source\n(n={:,})".format(
     nb_count("Inhibition", "source", "Metabolite")), "aux", 0.45, 0.15),
    ("Inhibition", "Conversion", "target\n(n={:,})".format(
     nb_count("Inhibition", "target", "Interaction")), "aux", -0.65, 0.4),
    ("Stimulation", "Metabolite", "source\n(n={:,})".format(
     nb_count("Stimulation", "source", "Metabolite")), "aux", 0.05, 0.5),
    ("Stimulation", "Conversion", "target\n(n={:,})".format(
     nb_count("Stimulation", "target", "Interaction")), "aux", -0.35, 0.4),
    ("TranscriptionTranslation", "GeneProduct", "source\n(n={:,})".format(
     nb_count("TranscriptionTranslation", "source", "GeneProduct")), "aux", 0.12, 0.5),
    ("TranscriptionTranslation", "Protein", "target\n(n={:,})".format(
     nb_count("TranscriptionTranslation", "target", "Protein")), "aux", 0.12, 0.5),
    ("Binding /\nComplexBinding", "Protein", "participants\n(n={:,})".format(
     nb_count("Binding", "participants", "Protein") + nb_count("ComplexBinding", "participants", "Protein")), "aux", 0.18, 0.5),
    ("Binding /\nComplexBinding", "DataNode", "participants\n(n={:,})".format(
     nb_count("Binding", "participants", "DataNode") + nb_count("ComplexBinding", "participants", "DataNode")), "aux", -0.18, 0.5),
]

color_map = {
    "feature":  "#a6d8a8",
    "target":   "#f4a3a3",
    "reaction": "#a3c9f4",
    "aux":      "#e6e6e6",
    "meta":     "#f4e3a3",
    "exclude":  "#fafafa",
}

role_styles = {
    "target":   dict(color="crimson", lw=2.5),
    "reaction": dict(color="dimgray", lw=1.0),
    "aux":      dict(color="dimgray", lw=1.0),
}

fig, ax = plt.subplots(figsize=(15, 11))
box_w, box_h = 2.2, 1.0
positions = {}
for label, (count, cat, (x, y)) in node_info.items():
    box = FancyBboxPatch((x - box_w / 2, y - box_h / 2), box_w, box_h,
                          boxstyle="round,pad=0.05", linewidth=1.3,
                          edgecolor="black", facecolor=color_map[cat], zorder=2)
    ax.add_patch(box)
    ax.text(x, y, f"{label}\n(n={count:,})", ha="center", va="center", fontsize=8.5, zorder=3)
    positions[label] = (x, y)

for src, dst, label, role, rad, t in edge_list:
    p0, p1 = positions[src], positions[dst]
    style = role_styles[role]
    arrow = FancyArrowPatch(p0, p1, arrowstyle="-|>", mutation_scale=14,
                             color=style["color"], lw=style["lw"],
                             connectionstyle=f"arc3,rad={rad}",
                             shrinkA=28, shrinkB=28, zorder=1)
    ax.add_patch(arrow)
    lx, ly = curve_point(p0, p1, rad, t)
    ax.text(lx, ly, label, fontsize=7, ha="center", va="center",
            color=style["color"], bbox=dict(facecolor="white", edgecolor="none", alpha=0.85, pad=1.2), zorder=4)

legend_items = [
    ("To featurise (03_*_encodings)", color_map["feature"]),
    ("Prediction target (Catalysis)", color_map["target"]),
    ("Reaction identity / context", color_map["reaction"]),
    ("Auxiliary context (not target)", color_map["aux"]),
    ("Pathway / Organism / External metadata", color_map["meta"]),
    ("Excluded from learning graph", color_map["exclude"]),
]
legend_patches = [mpatches.Patch(facecolor=c, edgecolor="black", label=l) for l, c in legend_items]
ax.legend(handles=legend_patches, loc="lower center", bbox_to_anchor=(0.5, -0.08),
          ncol=3, fontsize=8.5, framealpha=0.95)

ax.set_xlim(-1.7, 13.7)
ax.set_ylim(-3, 10.7)
ax.axis("off")
ax.set_title("PlantMetWiki heterogeneous graph schema (Interaction split by recovered subtype) -- AS OF THE ORIGINAL v2 GRAPH (pre-Section-9 redesign); DataNode/External read 0 if already redesigned")
plt.tight_layout()
plt.savefig(FIG_DIR / "graph_schema.png", dpi=150, bbox_inches="tight")
plt.show()

**Figure -- PlantMetWiki heterogeneous graph schema.** Each box is a node type
(for `Interaction`, a recovered subtype from Section 6) labelled with its node count `n`. Arrows
are the "reaction structure" edges (`source` / `target` / `participants`) with their edge
counts; red arrows mark the `Catalysis` edges that form the link-prediction target
`(Protein, catalyzes, Interaction)` (Section 8). `Pathway`, `Organism`, and `External` are shown
as boxes only -- their `is_part_of` / `organism` / `is_about` edges are omitted from the figure
for readability and summarised in the table below instead. Box colours indicate each type's role
in the modelling plan: green = entities to featurise, red = prediction target, blue = reaction
identity/context, grey = auxiliary context (kept but not the split target), yellow =
pathway/organism/external metadata, near-white = excluded from the learning graph.

In [ ]:
from IPython.display import Markdown, display

pathway_edges = edges[(edges["rel"] == "is_part_of") & (edges["dst_type"] == "Pathway")].copy()

int_pw = pathway_edges[pathway_edges["src_type"] == "Interaction"].copy()
int_pw["subtype"] = int_pw["src"].map(subtype_lookup).fillna("(generic, no leaf subtype)")
int_pw_counts = int_pw.groupby("subtype").size()

entity_pw_counts = (
    pathway_edges[pathway_edges["src_type"] != "Interaction"]
    .groupby("src_type").size()
)

org_edges = edges[(edges["rel"] == "organism") & (edges["dst_type"] == "Organism")]
org_counts = org_edges.groupby("src_type").size()

# (table row label, subtype key(s) into int_pw_counts) for the Interaction rows
SUBTYPE_ROWS = [
    ("Interaction: Conversion", "Conversion"),
    ("Interaction: Catalysis", "Catalysis"),
    ("Interaction: TranscriptionTranslation", "TranscriptionTranslation"),
    ("Interaction: Inhibition", "Inhibition"),
    ("Interaction: Stimulation", "Stimulation"),
    ("Interaction: Binding / ComplexBinding", ["Binding", "ComplexBinding"]),
    ("Interaction: generic (no leaf subtype)", "(generic, no leaf subtype)"),
]

pw_rows = []
for row_label, key in SUBTYPE_ROWS:
    keys = key if isinstance(key, list) else [key]
    pw_rows.append((row_label, sum(int(int_pw_counts.get(k, 0)) for k in keys)))
for src_type in ["Metabolite", "Protein", "GeneProduct", "DataNode"]:
    pw_rows.append((src_type, int(entity_pw_counts.get(src_type, 0))))
pw_total = sum(c for _, c in pw_rows)

org_rows = [(src_type, int(org_counts.get(src_type, 0)))
            for src_type in ["Protein", "GeneProduct", "Metabolite", "DataNode"]]
org_total = sum(c for _, c in org_rows)


def to_table(rows, total, col2):
    lines = [f"| Source node type | {col2} |", "|---|---:|"]
    lines += [f"| {name} | {cnt:,} |" for name, cnt in rows]
    lines.append(f"| **Total** | **{total:,}** |")
    return "\n".join(lines)


display(Markdown(
    "**`is_part_of` edges to `Pathway`** (used by the pathway-balanced split, "
    "`02_build_splits.ipynb` Section 6):\n\n"
    + to_table(pw_rows, pw_total, "Edges to Pathway")
    + "\n\n**`organism` edges to `Organism`** (used by the taxa split, "
    "`02_build_splits.ipynb` Section 5):\n\n"
    + to_table(org_rows, org_total, "Edges to Organism")
))

In [ ]:
core_node_info = {
    "Metabolite": (Metabolite_count, "feature",  (0.5, 3.0)),
    "Protein":    (Protein_count,    "feature",  (8.5, 3.0)),
    "Conversion": (n_conversion,     "reaction", (3.0, 5.0)),
    "Catalysis":  (n_catalysis,      "target",   (6.0, 5.0)),
}

core_edge_list = [
    ("Conversion", "Metabolite", "participants\n(substrate/product)",
     nb_count("Conversion", "participants", "Metabolite"), "reaction", 0.12, 0.5),
    ("Catalysis", "Protein", "source\n(enzyme)",
     nb_count("Catalysis", "source", "Protein"), "target", 0.12, 0.5),
    ("Catalysis", "Conversion", "target\n(reaction catalysed)",
     nb_count("Catalysis", "target", "Interaction"), "target", -0.6, 0.5),
]

fig, ax = plt.subplots(figsize=(8, 6))
positions = {}
for label, (count, cat, (x, y)) in core_node_info.items():
    box = FancyBboxPatch((x - box_w / 2, y - box_h / 2), box_w, box_h,
                          boxstyle="round,pad=0.05", linewidth=1.3,
                          edgecolor="black", facecolor=color_map[cat], zorder=2)
    ax.add_patch(box)
    ax.text(x, y, f"{label}\n(n={count:,})", ha="center", va="center", fontsize=9, zorder=3)
    positions[label] = (x, y)

for src, dst, label, count, role, rad, t in core_edge_list:
    p0, p1 = positions[src], positions[dst]
    style = role_styles[role]
    arrow = FancyArrowPatch(p0, p1, arrowstyle="-|>", mutation_scale=14,
                             color=style["color"], lw=style["lw"],
                             connectionstyle=f"arc3,rad={rad}",
                             shrinkA=30, shrinkB=30, zorder=1)
    ax.add_patch(arrow)
    lx, ly = curve_point(p0, p1, rad, t)
    ax.text(lx, ly, f"{label}\n(n={count:,})", fontsize=8, ha="center", va="center",
            color=style["color"], bbox=dict(facecolor="white", edgecolor="none", alpha=0.85, pad=1.5), zorder=4)

legend_patches_core = [
    mpatches.Patch(facecolor=color_map["feature"], edgecolor="black", label="To featurise (03_*_encodings)"),
    mpatches.Patch(facecolor=color_map["target"], edgecolor="black", label="Prediction target (Catalysis)"),
    mpatches.Patch(facecolor=color_map["reaction"], edgecolor="black", label="Reaction identity / context"),
]
ax.legend(handles=legend_patches_core, loc="lower center", bbox_to_anchor=(0.5, -0.18),
          ncol=3, fontsize=9, framealpha=0.95)

ax.set_xlim(-1.5, 10.5)
ax.set_ylim(1.8, 7.3)
ax.axis("off")
ax.set_title("Core schema for the Catalysis link-prediction task")
plt.tight_layout()
plt.savefig(FIG_DIR / "graph_schema_core.png", dpi=150, bbox_inches="tight")
plt.show()

**Figure -- Core schema for the `Catalysis` link-prediction task.** The minimal
subgraph directly relevant to the prediction target: `Catalysis` interactions link a `Protein`
(the enzyme, `source`) to a `Conversion` (the reaction, `target`), and `Conversion` interactions
link to their `Metabolite` participants (substrates/products), which give the reaction its
chemical identity. This is the subgraph that `(Protein, catalyzes, Interaction)` link prediction
(Section 8) operates over; `GeneProduct`, `DataNode`, the other `Interaction` subtypes, `Pathway`,
`Organism`, and `External` (all shown in the full schema above) provide additional context but
are not part of this core loop.

## 8. Prediction task: reconstructing pathways via `Catalysis` edges

**Task framing — enzyme discovery (Conversion → Protein direction).**
The task is to predict missing `(Protein, catalyzes, Conversion)` edges, answering the
question: *given a biochemical reaction, which protein catalyses it?*

This framing is motivated by the annotation asymmetry in the data:

| | Count | Share |
|--|--|--|
| Conversion (reaction) nodes | 84,470 | 100% |
| Reactions with ≥1 known catalyst | 4,572 | **5.4%** |
| **Reactions with NO known catalyst (orphan)** | **79,898** | **94.6%** |
| Protein nodes | 13,682 | 100% |
| Proteins annotated as any enzyme | 3,456 | **25.3%** |
| Proteins with NO catalysis annotation | 10,226 | **74.7%** |
| (Protein, catalyzes, Conversion) positive edges | 11,980 | — |

The database encodes the reactions (substrates, products, EC class) far more completely than
it encodes which genes catalyse them. The model is meant to fill this enzyme-assignment gap,
especially for under-characterised organisms not in the training set.

**Why Conversion → Protein and not the reverse?**
The alternative — *given a protein, which reactions does it catalyse?* (Protein → Conversion)
— is the protein function annotation problem. It is a valid question but not the bottleneck
here: the metabolic networks (reactions, pathways) are relatively well-documented; what is
missing is the enzyme gene assignment within each pathway.
In practice: a biochemist reconstructing a secondary metabolic pathway in a new plant species
starts from known reactions (the chemistry is conserved) and needs to find which of the
plant's genes encodes each enzyme. That is Framing A.

**Annotation sparsity of annotated reactions:**
Among the 4,572 reactions that do have a known catalyst, most have very few:
- 51.6% have exactly 1 known catalyst (one protein–reaction pair)
- 39.5% have 2–5 catalysts (isoenzymes or multi-species assignments)
- 8.9% have >5 catalysts (well-studied core reactions, e.g. glycolysis)
- Median: 1 catalyst per reaction · Mean: 2.62

**Cross-species generalisation is the core challenge.**
The knowledge graph covers 424 organisms. The taxa split (`02_build_splits.ipynb`)
holds out 11 val and 69 test organisms entirely from training — their proteins are
never seen during training. The model must use protein sequence (ESM-C 960-dim embeddings,
covering 61.7% of proteins) and reaction-chemistry features to transfer enzyme–reaction
knowledge from characterised to novel organisms. Embedding coverage is itself a bottleneck:
38.3% of proteins lack sequence embeddings, representing a hard ceiling on what the current
model can predict.

**Operationalisation.** The task is link prediction on the derived
`(Protein, catalyzes, Conversion)` edge type (11,980 positive pairs after collapsing the
reified `Catalysis` nodes — see Section 8.1 below). The ranking pool for evaluation is the
8,445 proteins that have ESM embeddings (`--embedded-only-ranking`); proteins without
embeddings are excluded because they are indistinguishable (all-zero vectors). The primary
evaluation metric is **P-H@K** (Protein Hits@K): for each val/test Conversion, is the
true catalyst in the top K of the 8,445 ranked proteins?

```
Random P-H@50 baseline  = 50 / 8,445 ≈ 0.59%
Best model so far (S1)  = P-H@50 = 8.67%  →  14.7× above random
Theoretical maximum     ≈ 71.3% (taxa val, limited by embedding coverage of true catalysts)
Theoretical max (S4)    = 100%   (--eval-embedded-positives-only: skips unevaluable edges)
```

This matches how WikiPathways pathways are authored: a `Conversion` is drawn first
(its substrate/product `Metabolite`s define what reaction it is), and separately annotated
with zero or more `Catalysis` edges saying *which enzyme performs it*. Reconstructing a
pathway means: given the `Conversion` nodes (known chemistry) and candidate `Protein` nodes,
recover the missing enzyme assignments.


In [ ]:
catalysis_ids = set(subtypes.loc[subtypes["subtype"] == "Catalysis", "node_id"])

cat_targets = edges[(edges["src"].isin(catalysis_ids)) & (edges["rel"] == "target")]
cat_sources = edges[(edges["src"].isin(catalysis_ids)) & (edges["rel"] == "source")]

n_conv_with_catalysis = cat_targets["dst"].nunique()
pct_conv_with_cat = 100 * n_conv_with_catalysis / n_conversion

n_enzymes = cat_sources["dst"].nunique()
pct_proteins_as_enzyme = 100 * n_enzymes / Protein_count

n_dn_connected = len(set(edges.loc[(edges["rel"] != "organism") & (edges["src_type"] == "DataNode"), "src"])
                      | set(edges.loc[(edges["rel"] != "organism") & (edges["dst_type"] == "DataNode"), "dst"]))
n_dn_isolated = DataNode_count - n_dn_connected

print(f"Conversions with >=1 Catalysis edge: {n_conv_with_catalysis:,} / {n_conversion:,} ({pct_conv_with_cat:.1f}%)")
print(f"  -> Conversions with NO known catalysing enzyme: {n_conversion - n_conv_with_catalysis:,} ({100 - pct_conv_with_cat:.1f}%)")
print(f"Distinct Proteins acting as a catalysing enzyme: {n_enzymes:,} / {Protein_count:,} ({pct_proteins_as_enzyme:.1f}%)")
if DataNode_count:
    print(f"DataNodes with only organism edges (or none): {n_dn_isolated:,} / {DataNode_count:,} "
          f"({100 * n_dn_isolated / DataNode_count:.1f}%)")
else:
    print("DataNode no longer exists as a node type (Section 9's redesign already applied).")

In [ ]:
from IPython.display import Markdown

dn_row = (
    f"| **`DataNode`** (n={DataNode_count:,}) | **Drop**, or at least exclude from message passing for this task. | "
    f"**{n_dn_isolated:,} / {DataNode_count:,} ({100 * n_dn_isolated / DataNode_count:.1f}%)** have *only* `organism` "
    f"edges -- essentially isolated from the reaction graph. |"
    if DataNode_count else
    "| **`DataNode`** | *(already dropped -- Section 9's v3 redesign)* | -- |"
)

display(Markdown(f"""
### Diagnosis: what to do with each edge type

| Edge type | Treatment | Reasoning |
|---|---|---|
| **`Catalysis`** (`Protein --source--> Catalysis --target--> Conversion`, n={n_catalysis:,}) | **Prediction target.** Collapse to a derived `(Protein, catalyzes, Conversion)` edge and split *that* into train/val/test. | The enzyme-reaction assignment. Only **{pct_conv_with_cat:.0f}%** of `Conversion`s currently have one -- the other **{100 - pct_conv_with_cat:.0f}%** ({n_conversion - n_conv_with_catalysis:,} reactions) are exactly the "missing reactions" to reconstruct. |
| **`Conversion` participants** (`Metabolite <-> Conversion`, n={nb_count("Conversion","participants","Metabolite"):,}) | **Context / input graph, always kept.** Used for message passing and to give each `Conversion` a "reaction identity" (its substrate/product set). | A `Conversion`'s metabolites define *what reaction it is*. Treating these as a second prediction target would conflate "which reaction is this" with "which enzyme does it" -- a different question. Could become a *separate* future task ("predict missing reaction participants"), but not this one. |
| **`TranscriptionTranslation`** (`GeneProduct --source--> ... --target--> Protein`, n={n_transcr:,}) | **Keep as an auxiliary `encodes` context edge** (`GeneProduct -> Protein`). | `Catalysis` only ever points at a `Protein`, never a `GeneProduct` (Section 6). This edge is the only link from `GeneProduct` into the reaction graph, so it lets gene-level features (`03_gene_encodings.ipynb`) reach the enzyme side. |
| **`Inhibition` / `Stimulation`** (`Metabolite --source--> ... --target--> Conversion`, n={n_inhib + n_stim:,}) | **Keep as a separate `regulates` context edge type, excluded from the target.** | Biologically real (allosteric regulation), but "this metabolite regulates this reaction" != "this metabolite is a substrate/product". Now that they're distinguishable via `subtype`, don't let them collapse into `Conversion`'s `participants` edges. |
| **`Binding` / `ComplexBinding`** (n={n_bind}) | **Drop.** | Negligible ({n_bind} nodes), adds a `Complex` concept for no real benefit at this stage. |
| **Generic `Interaction`** (diagram-only, n={n_generic:,}) | **Drop entirely** from `nodes.tsv` / `heterodata.pt`. | {100 * n_generic / n_interaction_total:.0f}% of "Interaction" nodes, almost no `source`/`target`/`participants` edges -- pure GPML layout artifacts. |
{dn_row}

### Concrete change for `02_build_splits.ipynb`

`TARGET_EDGE` is currently `("Protein", "is_part_of", "Interaction")` -- **{20645:,} edges that mix
three different subtypes**: `Catalysis` ({n_catalysis:,}, the enzyme-reaction link we actually
want), `TranscriptionTranslation` ({n_transcr:,}, gene expression), and `Stimulation` (a handful).
Splitting on this edge means a large share of "positive" examples in the current `splits.pt` are
actually "this protein is the product of this gene", not "this protein catalyses this reaction" --
a different task silently baked into the existing splits.

**Proposed new target**: filter `(Protein, is_part_of, Interaction)` to the `Interaction`
endpoints whose recovered `subtype == "Catalysis"` (n={n_catalysis:,} edges, covering
{pct_conv_with_cat:.0f}% of `Conversion` nodes) -- the "reconstruct which enzyme catalyses which
reaction" task. We'll carry the `interaction_subtype` label into `nodes.tsv` (via
`rdf_to_typed_tables.py`) so this filter is available to `02_build_splits.ipynb` without
re-parsing the RDF.

**Status as of Section 9**: this proposal has since been carried out and superseded by the fuller
`Protein -> Conversion` collapse -- see Section 8.1 and Section 9 below.
"""))

### 8.1 Confirming the collapse: `Catalysis` is structurally just a reified edge

The "Concrete change" above only proposed *filtering* the target edge to the Catalysis subtype
-- which is what `02_build_splits.ipynb` / `training/fulldata_baseline_common.py` actually
implemented (`(Protein, catalyzes, Interaction)` restricted to Catalysis-subtype `Interaction`
endpoints). It stopped one hop short of the fuller idea floated just above it: collapsing into a
direct `Protein -catalyzes-> Conversion` edge. Worth confirming that's actually safe before doing
it -- a `Catalysis` node could in principle attach to *multiple* Proteins (a multi-subunit
enzyme complex) or *multiple* Conversions (one catalytic event enabling several reactions), in
which case collapsing to a single dyadic edge would silently lose information. Checking the
exact cardinality on the real data, using the same `cat_sources`/`cat_targets` edge slices from
Section 8 above:

In [ ]:
cat_out_target = cat_targets.groupby("src").size()
cat_out_source = cat_sources.groupby("src").size()
print(f"Catalysis nodes with exactly 1 'target' edge (-> Conversion): {(cat_out_target == 1).sum():,} / {n_catalysis:,}")
print(f"Catalysis nodes with exactly 1 'source' edge (-> Protein):    {(cat_out_source == 1).sum():,} / {n_catalysis:,}")
print(f"Catalysis nodes with >1 of either (would block a clean collapse): "
      f"{int((cat_out_target > 1).sum() + (cat_out_source > 1).sum())}")
print()

prot_cat_conv = (
    cat_sources[["src", "dst"]].rename(columns={"dst": "protein"})
    .merge(cat_targets[["src", "dst"]].rename(columns={"dst": "conversion"}), on="src")
)
n_distinct_pairs = prot_cat_conv[["protein", "conversion"]].drop_duplicates().shape[0]
n_distinct_conv = prot_cat_conv["conversion"].nunique()

if n_distinct_conv == 0:
    print("Catalysis nodes no longer exist in the live graph (Section 9's v3 redesign already "
          "collapsed and removed them) -- nothing left to confirm here. This cardinality check "
          "is a historical record of the analysis that justified that collapse; see Section 9.2 "
          "for the equivalent check run against the pre-redesign snapshot for TranscriptionTranslation.")
    fan_in = pd.Series(dtype=int)
else:
    print(f"(Protein, Conversion) pairs if collapsed directly: {n_distinct_pairs:,} distinct pairs "
          f"from {len(prot_cat_conv):,} Catalysis instances")
    print(f"  -> only {len(prot_cat_conv) - n_distinct_pairs} Catalysis instances are exact duplicates "
          f"(same protein, same conversion, via a different Catalysis node) -- negligible data loss")
    print(f"Distinct Conversions touched: {n_distinct_conv:,} "
          f"(vs {n_catalysis:,} Catalysis instances today -- the H@K ranking pool would shrink "
          f"~{n_catalysis / n_distinct_conv:.1f}x, to a biologically meaningful pool of *reactions* "
          f"rather than *enzyme-reaction instance records*)")

    fan_in = prot_cat_conv.groupby("conversion").size()
    print()
    print("Catalysis instances per Conversion (fan-in -- different enzymes/isozymes/organisms catalysing the same reaction):")
    print(fan_in.value_counts().sort_index().rename("n_conversions").to_string())

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))
fan_in.value_counts().sort_index().plot(kind="bar", ax=ax, color="#4C78A8", edgecolor="black")
ax.set_xlabel("# Catalysis instances pointing at the same Conversion")
ax.set_ylabel("# Conversion nodes")
ax.set_title("Fan-in: how many Catalysis records collapse onto one reaction")
plt.tight_layout()
plt.savefig(FIG_DIR / "catalysis_conversion_fanin.png", dpi=150)
plt.show()
print(f"Saved -> {FIG_DIR}/catalysis_conversion_fanin.png")

**Result: turning `Catalysis` into a direct edge is structurally sound.** Every Catalysis node
has *exactly* one `source` (Protein) and one `target` (Conversion) -- no multi-subunit fan-out,
no multi-reaction fan-out. Chaining the existing `Protein -source-> Catalysis -target->
Conversion` paths into a single derived `(Protein, catalyzes, Conversion)` edge loses essentially
nothing (a handful of exact-duplicate instances, immaterial): 12,359 Catalysis instances collapse
to 11,980 distinct `(Protein, Conversion)` positive pairs (the cell above).

**Correction on candidate-pool size -- it does *not* shrink.** An earlier draft of this section
claimed collapsing shrinks the H@K ranking pool to the 4,572 Conversions that currently have a
known catalyst. That's the wrong comparison: today's pool (`ctx.cat_idxs`) is *every*
Catalysis-subtype node (12,359), not just ones with edges in a particular split -- the consistent
analogue after collapsing is *every* Conversion-subtype node, **19,927** of them (the cell below
the histogram confirms `n_conversion = 19,927`), not just the 4,572 with a known catalyst today.
So the pool actually **grows** ~1.6x, including the 15,355 Conversions with no known catalyst at
all -- which is arguably the more realistic evaluation anyway (ranking against the *full*
universe of known reactions, not a pre-filtered "has an answer" subset). The real, solid benefit
here is the deduplication (12,359 -> 11,980 positives) and removing near-duplicate target nodes
that differed only in *which* enzyme attaches to the *same* Conversion (the fan-in histogram
above) -- not a smaller pool.

**Why this also matters for message passing, not just deduplication.** `Conversion` nodes
connect directly to their substrate/product `Metabolite`s (`source`/`target`/`participants`,
Section 6/7 above) -- one hop from `Conversion`. In the *current* schema, that puts `Metabolite`
chemistry **three hops** from `Protein`:

```
Protein --source--> Catalysis (1 hop) --target--> Conversion (2 hops) --...--> Metabolite (3 hops)
```

`04_link_prediction.ipynb`'s model uses 2 `HeteroConv` layers (chosen specifically to reach a
Protein's 2-hop neighbourhood) -- so **reaction chemistry is currently outside the model's
receptive field entirely**; a Protein's embedding can see *that* it attaches to some Catalysis
node and *that* the Catalysis node attaches to some Conversion, but never *what that Conversion
actually converts*. Collapsing the indirection frees up exactly the hop that's being spent on
traversing the reified node:

```
Protein --catalyzes--> Conversion (1 hop) --...--> Metabolite (2 hops)
```

The same 2-layer budget now reaches substrate/product chemistry for the first time, with no
architecture change beyond redefining the target edge. This is a plausible structural
contributor to why Common Neighbours/Jaccard (unconstrained by any fixed layer budget --
they just count raw overlap) still beat the GNN in `04_link_prediction.ipynb` §8 -- though note
the *larger* candidate pool (above) works against the GNN here too, so any H@K change after the
redefinition reflects both effects at once, not message-passing depth alone.

**Recommendation, and what changes downstream:** derive `(Protein, catalyzes, Conversion)` by
chaining the existing `Protein -source-> Catalysis -target-> Conversion` paths (exactly the join
above), and use *that* as the link-prediction target edge everywhere instead of `(Protein,
catalyzes, Interaction)` restricted to Catalysis-subtype nodes. Concretely:
`02_build_splits.ipynb` derives and splits this edge instead (rebuilding `splits_taxa.pt`),
`training/fulldata_baseline_common.py`'s `load_taxa_splits()` derives it the same way (and its
H@K ranking pool becomes all Conversion-subtype nodes, not Catalysis-subtype), and
`04_link_prediction.ipynb` re-runs against it. The `Catalysis` node type itself doesn't need to
be deleted from `heterodata.pt` -- it simply stops being the prediction target.

In [ ]:
# Before/after: the current 3-hop indirection through Catalysis vs. the proposed
# direct 1-hop Protein-catalyzes->Conversion edge, with each node's hop-distance
# from Protein annotated -- this is what changes for message passing (cell above).
collapse_node_info = {
    "Protein":    (Protein_count,  "feature",  (0.3, 3.0)),
    "Catalysis":  (n_catalysis,    "target",   (3.3, 4.6)),
    "Conversion": (n_conversion,   "reaction", (6.3, 3.0)),
    "Metabolite": (Metabolite_count, "feature", (9.3, 3.0)),
}
hop_labels = {  # (hop via current indirect path, hop via proposed direct edge)
    "Protein": (0, 0), "Catalysis": (1, None), "Conversion": (2, 1), "Metabolite": (3, 2),
}

fig, ax = plt.subplots(figsize=(11, 5.5))
positions = {}
for label, (count, cat, (x, y)) in collapse_node_info.items():
    box = FancyBboxPatch((x - box_w / 2, y - box_h / 2), box_w, box_h,
                          boxstyle="round,pad=0.05", linewidth=1.3,
                          edgecolor="black", facecolor=color_map[cat], zorder=2)
    ax.add_patch(box)
    hop_cur, hop_new = hop_labels[label]
    hop_txt = f"hop {hop_cur}" if hop_new is None else f"hop {hop_cur} (current) / hop {hop_new} (proposed)"
    ax.text(x, y + 0.05, f"{label}\n(n={count:,})", ha="center", va="center", fontsize=9, zorder=3)
    ax.text(x, y - box_h / 2 - 0.28, hop_txt, ha="center", va="center", fontsize=7.5, style="italic", color="dimgray")
    positions[label] = (x, y)

# Current path: Protein -> Catalysis -> Conversion -> Metabolite (solid, crimson)
for src, dst, lbl in [("Protein", "Catalysis", "source\n(enzyme)"),
                       ("Catalysis", "Conversion", "target\n(reaction)"),
                       ("Conversion", "Metabolite", "participants\n(substrate/product)")]:
    p0, p1 = positions[src], positions[dst]
    ax.add_patch(FancyArrowPatch(p0, p1, arrowstyle="-|>", mutation_scale=14, color="crimson", lw=2.2,
                                  connectionstyle="arc3,rad=0.15", shrinkA=30, shrinkB=30, zorder=1))
    lx, ly = curve_point(p0, p1, 0.15, 0.5)
    ax.text(lx, ly + 0.35, lbl, fontsize=7.5, ha="center", color="crimson",
            bbox=dict(facecolor="white", edgecolor="none", alpha=0.85, pad=1.0), zorder=4)

# Proposed direct edge: Protein -> Conversion (dashed, green), skipping Catalysis
p0, p1 = positions["Protein"], positions["Conversion"]
ax.add_patch(FancyArrowPatch(p0, p1, arrowstyle="-|>", mutation_scale=14, color="#2E7D32", lw=2.2,
                              linestyle="--", connectionstyle="arc3,rad=-0.35", shrinkA=30, shrinkB=30, zorder=1))
lx, ly = curve_point(p0, p1, -0.35, 0.5)
ax.text(lx, ly - 0.25, "catalyzes (proposed, direct)", fontsize=8, ha="center", color="#2E7D32", weight="bold",
        bbox=dict(facecolor="white", edgecolor="none", alpha=0.85, pad=1.0), zorder=4)

ax.axvline(positions["Conversion"][0] + 1.6, color="gray", linestyle=":", lw=1)
ax.text(positions["Conversion"][0] + 1.7, 5.4, "2-layer GNN\nreceptive-field edge", fontsize=7.5, color="gray", va="top")

legend_patches = [
    mpatches.Patch(facecolor="none", edgecolor="crimson", label="Current: Protein -> Catalysis -> Conversion (indirect, 2 hops)"),
    mpatches.Patch(facecolor="none", edgecolor="#2E7D32", label="Proposed: Protein -> Conversion (direct, 1 hop)"),
]
ax.legend(handles=legend_patches, loc="lower center", bbox_to_anchor=(0.5, -0.22), ncol=1, fontsize=8.5, framealpha=0.95)

ax.set_xlim(-1.5, 11.5)
ax.set_ylim(1.0, 6.0)
ax.axis("off")
ax.set_title("Collapsing Catalysis: same biology, one fewer hop to reaction chemistry")
plt.tight_layout()
plt.savefig(FIG_DIR / "catalysis_collapse_before_after.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved -> {FIG_DIR}/catalysis_collapse_before_after.png")

**Figure -- collapsing `Catalysis` into a direct edge.** Crimson: the current path from
`Protein` to reaction chemistry, through the reified `Catalysis` node (2 hops to `Conversion`,
3 to `Metabolite`). Green dashed: the proposed direct `(Protein, catalyzes, Conversion)` edge,
which reaches the same `Conversion` in 1 hop and its `Metabolite` substrates/products in 2 --
within a 2-layer GNN's receptive field for the first time. This is implemented from
`02_build_splits.ipynb` onward (see Recommendation above); the run in
`04_link_prediction.ipynb` is re-done against this new target and documented there as a task
redefinition, not a like-for-like rerun of the same prediction problem.

### Using `Organism` beyond the taxa split

The taxa split (`02_build_splits.ipynb` Section 5) already uses `(Protein, organism, Organism)`
edges to hold out whole species. The same edges -- already present in `heterodata.pt` for
`Protein`, `GeneProduct`, `Metabolite`, and `DataNode` (see the schema figure above) -- could also
inform the model itself:

- **`Organism` as a node type for message passing** -- zero extra pipeline work, since the edges
  already exist in `heterodata.pt`. Proteins from the same species would share an `Organism`
  neighbour, which is a reasonable inductive bias: homologous enzymes across related species often
  catalyse the same reaction.
- **`Organism` as a categorical feature** for `Protein` / `GeneProduct` / `Metabolite` in
  `03_*_encodings.ipynb` -- also reasonable and low-effort (a one-hot or learned embedding over the
  ~424 organism IDs).

**Caveat for the taxa-held-out split specifically**: feeding organism identity directly into the
model risks the model leaning on "which species is this" as a shortcut rather than learning
enzyme-level features -- which would undercut exactly what that split is meant to test
(generalisation to unseen species). For the random and pathway-balanced splits this is not a
concern. See `02_build_splits.ipynb` Section 5.5 for a worked-through discussion of this trade-off.

## 9. Graph redesign (v3): removing noise node types, collapsing reified edges

Following on from Section 8's `Catalysis` collapse, four further structural changes were made
directly to `data/processed/nodes.tsv` / `edges.tsv` (and so to `heterodata.pt`) by
`scripts/redesign_graph_v3.py`, run once as a graph-build step (not at training time, unlike the
`Catalysis` collapse above, which several scripts used to re-derive on every load):

1. **`DataNode` (42,493 nodes)** -- generic, pathway-local diagram elements, never deduplicated
   across pathways (Section 6) -- classified by local-identifier pattern into `GeneProduct` /
   `Protein` / `Metabolite` / unclassifiable. Classified entries whose identifier exactly matches
   an existing canonical node of the same type are **merged** into it (edges redirected, duplicate
   dropped -- recovers a real pre-computed embedding for free); the rest are **retyped** in place,
   keeping the same `node_id`. Unclassifiable entries are dropped.
2. **`External` (70 nodes)** -- genuinely untyped leftovers (Section 6's `is_about` mislabelling
   artifact) -- dropped entirely.
3. **`Inhibition` / `Stimulation` / `ComplexBinding`** Interaction subtypes (3,486 + 865 + 70 =
   4,421 nodes) -- dropped entirely, not collapsed to an edge (unlike Catalysis/TranscriptionTranslation
   below, there is no natural single dyadic edge these reduce to without losing information, and they
   were not requested to be kept).
4. **`Catalysis` (12,359 nodes)** and **`TranscriptionTranslation` (8,280 nodes)** -- both reified
   relation nodes -- collapsed into direct, *persisted* edges (`Protein, catalyzes, Interaction` /
   `GeneProduct, encodes, Protein`, each with a reverse edge), then removed as nodes. `Catalysis`
   was already shown structurally sound to collapse in Section 8.1; the same check for
   `TranscriptionTranslation` is below (9.2).

`Conversion`-subtype and blank-subtype Interaction nodes are untouched.

The pre-redesign tables were snapshotted to `data/processed_v2_backup/` before any of this ran, so
the diagnostics below load *that* snapshot to show a real before/after, rather than just asserting
one (the live `nodes`/`edges`/`data` loaded in cell 2 are already the *post*-redesign graph).

### 9.1 `DataNode` classification heuristic, and what it recovers vs. drops

In [ ]:
sys.path.insert(0, str(ROOT / "scripts"))
from redesign_graph_v3 import classify_datanode, COMMON_METABOLITE_NAMES, DROP_INTERACTION_SUBTYPES  # noqa: E402

BACKUP_DIR = ROOT / "data" / "processed_v2_backup"
nodes_v2 = pd.read_csv(BACKUP_DIR / "nodes.tsv", sep="\t")

dn_v2 = nodes_v2.loc[nodes_v2["node_type"] == "DataNode", ["node_id"]].copy()
dn_v2["suffix"] = dn_v2["node_id"].str.rsplit("/", n=1).str[-1]
dn_v2["new_type"] = dn_v2["suffix"].apply(classify_datanode)

bucket_counts = dn_v2["new_type"].fillna("DROP (unclassifiable)").value_counts()
print(f"DataNode: {len(dn_v2):,} total")
print(bucket_counts.to_string())
print()

canon_v2 = nodes_v2.loc[nodes_v2["node_type"].isin(["Protein", "GeneProduct", "Metabolite"]), ["node_id", "node_type"]].copy()
canon_v2["suffix"] = canon_v2["node_id"].str.rsplit("/", n=1).str[-1]
canon_lookup_v2 = {(t, s) for t, s in zip(canon_v2["node_type"], canon_v2["suffix"])}
dn_v2["merged"] = [
    (nt, s) in canon_lookup_v2 for nt, s in zip(dn_v2["new_type"], dn_v2["suffix"])
]
n_merged = int((dn_v2["merged"] & dn_v2["new_type"].notna()).sum())
print(f"Of the classified entries, {n_merged:,} exactly match an existing canonical node's "
      f"identifier of the same type -- these get merged into it instead of becoming a new node.")
print()

print("Sample suffixes per bucket:")
for bucket in ["Protein", "GeneProduct", "Metabolite", None]:
    sample = dn_v2.loc[dn_v2["new_type"] == bucket, "suffix"].sample(min(8, (dn_v2["new_type"] == bucket).sum()), random_state=7)
    label = bucket if bucket else "DROP (unclassifiable)"
    print(f"  {label}: {sample.tolist()}")

### 9.2 Confirming the collapse: `TranscriptionTranslation` cardinality

Same check as Section 8.1 did for `Catalysis`, now for `TranscriptionTranslation`: does every
instance have exactly one `GeneProduct` source and one `Protein` target, so collapsing to a direct
`(GeneProduct, encodes, Protein)` edge loses nothing?

In [ ]:
edges_v2 = pd.read_csv(BACKUP_DIR / "edges.tsv", sep="\t")
tt_ids = set(nodes_v2.loc[nodes_v2["interaction_subtype"] == "TranscriptionTranslation", "node_id"])
n_tt = len(tt_ids)

tt_part_of = edges_v2[
    (edges_v2["rel"] == "is_part_of") & (edges_v2["src_type"] == "GeneProduct") & (edges_v2["dst"].isin(tt_ids))
]
tt_targets = edges_v2[
    (edges_v2["rel"] == "target") & (edges_v2["src_type"] == "Interaction") & (edges_v2["src"].isin(tt_ids))
    & (edges_v2["dst_type"] == "Protein")
]

gp_out = tt_part_of.groupby("dst").size()
prot_out = tt_targets.groupby("src").size()
print(f"TranscriptionTranslation nodes: {n_tt:,}")
print(f"  with exactly 1 GeneProduct source: {(gp_out == 1).sum():,} / {n_tt:,}  "
      f"({n_tt - len(gp_out):,} have none at all, not multiple)")
print(f"  with >1 GeneProduct source (would block a clean collapse): {int((gp_out > 1).sum())}")
print(f"  with exactly 1 Protein target: {(prot_out == 1).sum():,} / {n_tt:,}")
print(f"  with >1 Protein target (would block a clean collapse): {int((prot_out > 1).sum())}")

gp_to_tt = dict(zip(tt_part_of["dst"], tt_part_of["src"]))
tt_to_prot = dict(zip(tt_targets["src"], tt_targets["dst"]))
pairs = {(gp, tt_to_prot[tt]) for gp, tt in gp_to_tt.items() if tt in tt_to_prot}
print(f"\n(GeneProduct, Protein) pairs if collapsed directly: {len(pairs):,} distinct pairs "
      f"from {len(gp_to_tt):,} TranscriptionTranslation instances with a GeneProduct source")
print(f"  -> {n_tt - len(gp_to_tt):,} instances have no GeneProduct source at all (no edge to derive from them)")

### 9.3 Before / after

In [ ]:
before_counts = nodes_v2["node_type"].value_counts()
after_counts = nodes["node_type"].value_counts()  # `nodes` from cell 2 -- the live, post-redesign table
node_compare = pd.DataFrame({"before (v2)": before_counts, "after (v3)": after_counts}).fillna(0).astype(int)
node_compare = node_compare.reindex(
    sorted(set(before_counts.index) | set(after_counts.index), key=lambda t: -node_compare.loc[t].sum())
)
print(node_compare.to_string())
print()
print(f"Total nodes: {before_counts.sum():,} -> {after_counts.sum():,}")
print(f"Total edges: {len(edges_v2):,} -> {len(edges):,}")
print()
print(f"(Protein, catalyzes, Interaction): {data[('Protein', 'catalyzes', 'Interaction')].edge_index.shape[1]:,} persisted edges")
print(f"(GeneProduct, encodes, Protein):   {data[('GeneProduct', 'encodes', 'Protein')].edge_index.shape[1]:,} persisted edges")

### 9.4 Known limitations / not done in this pass

- **`DataNode` classification is a heuristic, not an identity resolution.** It recognises locus
  tags (`AT#G#####`, `Glyma...`), the PlantCyc `-MONOMER` protein convention, `CPD-####` compound
  IDs, and a short list of ultra-common cofactors named by common name rather than catalog ID
  (`WATER`, `PROTON`, `ATP`, ...). It does **not** attempt name-to-structure resolution for the
  remaining metabolite common names (e.g. `LATHYRINE`, `GLUTACONATE`), or recognise locus-tag
  conventions from genome projects beyond Arabidopsis/soybean (e.g. `ZM00001EB222680` for maize,
  `PAVIR_7KG337500` for switchgrass) -- these fall into the unclassifiable/dropped bucket. This
  mirrors a pre-existing limitation in the embedding pipeline itself
  (`03_embeddings_fulldata.ipynb` §4.3's locus-extraction coverage), so little real signal is lost
  beyond what was already unreachable.
- **No new embeddings were computed for retyped/merged nodes in this pass.** Merged entries
  (~2,639) inherit whatever embedding the canonical node they merged into already had. Retyped
  entries (~39,854) are new Protein/GeneProduct/Metabolite nodes with no embedding yet --
  `03_embeddings_fulldata.ipynb`'s embedding-attachment step gives them the same zero-vector
  fallback any other unresolved node of that type already gets. Attempting real resolution for
  the retyped Protein/GeneProduct entries (many share the same locus-tag/UniProt-style
  identifiers the existing resolver already knows how to look up) is a natural next step, not
  done here.
- **Blank-subtype Interaction nodes (~64,543, diagram-only connectors, Section 6) are untouched.**
  Only `Inhibition`/`Stimulation`/`ComplexBinding` were dropped, per explicit scope -- the blank
  ones were not requested to be removed, even though they share some of the same "mostly inert"
  character (Section 8 above).
- **The `(Protein, catalyzes, Interaction)` edge count is unchanged at 11,980** -- the same as the
  v2 runtime-derived count (`fb8342d4` above) -- because `scripts/redesign_graph_v3.py` collapses
  `Catalysis` using `edges.tsv`'s `src_type` column *before* it gets recomputed from the
  post-retype node table (the recompute happens afterwards, see the script's step 6/7), so
  retyped DataNode-Proteins' `is_part_of` edges to `Catalysis` nodes -- if any exist -- are not
  picked up by this collapse. Checking directly: there are in fact **zero** such edges in this
  graph, so it makes no difference here, but it is worth knowing the collapse order doesn't
  retroactively benefit from the `DataNode` reclassification, in case a future change introduces
  some.
- **Self-loop bug caught and fixed during development**: an earlier version of
  `redesign_graph_v3.py`'s `is_part_of` filter forgot to restrict by source type, so it also
  picked up `Interaction -is_part_of-> Interaction` edges (a different, unrelated GPML nesting
  relation) and produced spurious Conversion-catalyzes-itself self-loops. Caught by checking for
  `src == dst` in the rebuilt `edges.tsv` before this was merged in.